In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
clusters = 4

gex_methods = ['demuxlet', 'freemuxlet', 'vireo', 'vireo_nogenos',
               'scsplit', 'scsplit_nogenos', 'souporcell',
               'souporcell_nogenos', 'demuxalot', 'ambimux', 'ambimux_joint']
atac_methods = ['demuxlet', 'freemuxlet', 'vireo', 'vireo_nogenos', 
            'scsplit', 'scsplit_nogenos', 'souporcell',
       'souporcell_nogenos', 'scavengers', 'ambimux', 'ambimux_joint']

overlap_methods = ['demuxlet', 'freemuxlet', 'vireo', 'vireo_nogenos', 'scsplit', 'scsplit_nogenos',
                   'souporcell', 'souporcell_nogenos', 'ambimux', 'ambimux_joint']

In [ ]:
inter_gex = pd.read_csv(f'{projdir}/csv/demux/inter_gex.csv', sep='\t', header=0, index_col=0)
inter_atac = pd.read_csv(f'{projdir}/csv/demux/inter_atac.csv', sep='\t', header=0, index_col=0)

In [ ]:
ambimux_gex = pd.read_csv(f'{projdir}/csv/demux/gex/ambimux.csv', sep='\t', header=0, index_col=0)
ambimux_atac = pd.read_csv(f'{projdir}/csv/demux/atac/ambimux.csv', sep='\t', header=0, index_col=0)
ambimux_joint = pd.read_csv(f'{projdir}/csv/demux/ambimux.csv', sep='\t', header=0, index_col=0)
ambimux_gex.shape, ambimux_atac.shape, ambimux_joint.shape

In [ ]:
inter_gex['ambimux'] = ambimux_gex['assignment']
inter_atac['ambimux'] = ambimux_atac['assignment']
inter_gex['ambimux_joint'] = ambimux_joint['assignment']
inter_atac['ambimux_joint'] = ambimux_joint['assignment']

In [ ]:
singlet_gex = inter_gex.replace({'doublet' : 'non_singlet',
                                'unassigned' : 'non_singlet'})
singlet_atac = inter_atac.replace({'doublet' : 'non_singlet',
                                'unassigned' : 'non_singlet'})

mask = singlet_gex != 'non_singlet'
singlet_gex[mask] = 'singlet'

mask = singlet_atac != 'non_singlet'
singlet_atac[mask] = 'singlet'
    
tmp_singlet_gex = singlet_gex.copy()
tmp_singlet_gex.replace({'non_singlet' : 0,
                         'singlet' : 1}, inplace=True)
    
tmp_singlet_atac = singlet_atac.copy()
tmp_singlet_atac.replace({'non_singlet' : 0,
                         'singlet' : 1}, inplace=True)

tmp_singlet_gex.to_csv(f'{projdir}/csv/demux/singlet_gex.csv', sep='\t', header=True, index=True)
tmp_singlet_atac.to_csv(f'{projdir}/csv/demux/singlet_atac.csv', sep='\t', header=True, index=True)
tmp_singlet_gex.columns = [f'gex_{x}' for x in tmp_singlet_gex.columns]
tmp_singlet_atac.columns = [f'atac_{x}' for x in tmp_singlet_atac.columns]
tmp_singlet_merged = pd.concat([tmp_singlet_gex, tmp_singlet_atac], axis=1)
tmp_singlet_merged.to_csv(f'{projdir}/csv/demux/singlet_merged.csv', sep='\t', header=True, index=True)

In [ ]:
status_gex = inter_gex.copy()
status_atac = inter_atac.copy()

status_gex.replace({'doublet' : 'non_singlet',
                  'unassigned' : 'non_singlet'}, inplace=True)

status_atac.replace({'doublet' : 'non_singlet',
                  'unassigned' : 'non_singlet'}, inplace=True)

tmp_status_gex = status_gex.copy()
tmp_status_atac = status_atac.copy()

tmp_status_gex.to_csv(f'{projdir}/csv/demux/status_gex.csv', sep='\t', header=True, index=True)
tmp_status_atac.to_csv(f'{projdir}/csv/demux/status_atac.csv', sep='\t', header=True, index=True)
tmp_status_gex.columns = [f'gex_{x}' for x in tmp_status_gex.columns]
tmp_status_atac.columns = [f'atac_{x}' for x in tmp_status_atac.columns]
tmp_status_merged = pd.concat([tmp_status_gex, tmp_status_atac], axis=1)
tmp_status_merged.to_csv(f'{projdir}/csv/demux/status_merged.csv', sep='\t', header=True, index=True)

In [ ]:
tmp_gex = status_gex.melt()
tmp_atac = status_atac.melt()

In [ ]:
status_gex.columns

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 6))
sns.set_theme(palette="husl", style='whitegrid')

plot_order = donors + ['non_singlet']
ax = sns.histplot(data=tmp_gex, x='variable', hue='value', multiple='stack',hue_order=plot_order)
ax.set_title('Assignments, RNA', fontsize=18)
ax.grid(False)
ax.set_xticklabels(status_gex.columns, rotation=45, fontsize=15)
ax.set_xlabel('')
ax.set_ylabel('')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1., 1), title='donor')

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 6))

sns.set_theme(palette="husl", style='whitegrid')

plot_order = donors + ['non_singlet']
ax = sns.histplot(data=tmp_atac, x='variable', hue='value', multiple='stack',hue_order=plot_order)
ax.set_title('Assignments, ATAC', fontsize=18)
ax.grid(False)
ax.set_xticklabels(status_atac.columns, rotation=45, fontsize=15)
ax.set_xlabel('')
ax.set_ylabel('')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1., 1), title='donor')

In [ ]:
### plot with RNA/ATAC in name

In [ ]:
final_gex = tmp_gex.copy()
final_gex['variable'] = [f'{x}_RNA' for x in final_gex['variable']]
final_atac = tmp_atac.copy()
final_atac['variable'] = [f'{x}_ATAC' for x in final_atac['variable']]

In [ ]:
final_dist = pd.concat([final_gex, final_atac])

In [ ]:
sorted_final_dist = final_dist.sort_values(by='variable')

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 6))
xlabels = final_dist['variable'].unique()

sns.set_theme(palette="husl", style='whitegrid')

plot_order = donors + ['non_singlet']
ax = sns.histplot(data=final_dist, x='variable', hue='value', multiple='stack',hue_order=plot_order)
ax.set_title('Assignments, RNA/ATAC', fontsize=18)
ax.grid(False)
ax.set_xticklabels(xlabels, rotation=45, fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1., 1), title='donor')

In [ ]:
fig, axes = plt.subplots(1, figsize=(12, 10))
xlabels = sorted(sorted_final_dist['variable'].unique())

sns.set_theme(palette="husl", style='whitegrid')

plot_order = donors + ['non_singlet']
ax = sns.histplot(data=sorted_final_dist, x='variable', hue='value', multiple='stack',hue_order=plot_order)
ax.set_title('Assignments, RNA/ATAC', fontsize=18)
ax.grid(False)
ax.set_xticklabels(xlabels, rotation=45, fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1., 1), title='donor')

In [ ]:
final_dist

In [ ]:
singlet_gex.columns

In [ ]:
n_drops = singlet_gex.shape[0]
tmp_gex_methods = singlet_gex.columns[:-1]
all_overlap_gex = pd.DataFrame(columns=tmp_gex_methods)
for method1 in tmp_gex_methods:
    all_overlap_gex.loc[method1] = [np.sum(singlet_gex[method1] == singlet_gex[method2]) / n_drops
                            for method2 in tmp_gex_methods]
all_overlap_gex

In [ ]:
n_drops = singlet_atac.shape[0]
tmp_atac_methods = singlet_atac.columns[:-1]
all_overlap_atac = pd.DataFrame(columns=tmp_atac_methods)
for method1 in tmp_atac_methods:
    all_overlap_atac.loc[method1] = [np.sum(singlet_atac[method1] == singlet_atac[method2]) / n_drops
                            for method2 in tmp_atac_methods]
all_overlap_atac

In [ ]:
inter_gex = pd.read_csv(f'{projdir}/csv/demux/inter_gex.csv', sep='\t', header=0, index_col=0)
inter_atac = pd.read_csv(f'{projdir}/csv/demux/inter_atac.csv', sep='\t', header=0, index_col=0)
inter_gex['sample'] = [''.join(x.split('_')[1:]) for x in inter_gex.index]
inter_atac['sample'] = [''.join(x.split('_')[1:]) for x in inter_atac.index]

In [ ]:
singlet_gex['sample'] = inter_gex['sample']
tmp_gex_methods = singlet_gex.columns[:-1]
gex_overlap = []
for sample in samples:
    tmp_gex = singlet_gex[singlet_gex['sample'] == sample].drop(columns=['sample'])
    n_drops = tmp_gex.shape[0]
    overlap = pd.DataFrame(columns=tmp_gex_methods)
    for method1 in tmp_gex_methods:
        overlap.loc[method1] = [np.sum(tmp_gex[method1] == tmp_gex[method2]) / n_drops
                                for method2 in tmp_gex_methods]
    gex_overlap.append(overlap)

In [ ]:
singlet_atac['sample'] = inter_atac['sample']
tmp_atac_methods = singlet_atac.columns[:-1]
atac_overlap = []
for sample in samples:
    tmp_atac = singlet_atac[singlet_atac['sample'] == sample].drop(columns=['sample'])
    n_drops = tmp_atac.shape[0]
    overlap = pd.DataFrame(columns=tmp_atac_methods)
    for method1 in tmp_atac_methods:
        overlap.loc[method1] = [np.sum(tmp_atac[method1] == tmp_atac[method2]) / n_drops
                                for method2 in tmp_atac_methods]
    atac_overlap.append(overlap)

### plot per sample

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(30, 8))  # 2 rows, 4 columns
plt.subplots_adjust(wspace=1.0, hspace=0.5)
for i, ax in enumerate(axes.flat):
    tmp_overlap = gex_overlap[i]
    s = samples[i]
    sns.heatmap(tmp_overlap, annot=True, xticklabels='', vmin=0, vmax=1, cmap='bwr',
        annot_kws={'size': 8}, cbar=False, ax = ax)
    ax.set_title(s, fontsize=20)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(30, 8))  # 2 rows, 4 columns
plt.subplots_adjust(wspace=1.0, hspace=0.5)
for i, ax in enumerate(axes.flat):
    tmp_overlap = atac_overlap[i]
    s = samples[i]
    sns.heatmap(tmp_overlap, annot=True, xticklabels='', vmin=0, vmax=1, cmap='bwr',
        annot_kws={'size': 8}, cbar=False, ax = ax)
    ax.set_title(s, fontsize=20)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 14))  # 2 rows, 4 columns
plt.subplots_adjust(wspace=0.4)

sns.heatmap(all_overlap_gex, annot=True, xticklabels='', vmin=0, vmax=1, cmap='bwr',
            annot_kws={'size': 17}, cbar=False, ax = axes[0])
sns.heatmap(all_overlap_atac, annot=True, xticklabels='', vmin=0, vmax=1, cmap='bwr',
            annot_kws={'size': 17}, cbar=False, ax = axes[1])

axes[0].set_title('RNA', fontsize=20)
axes[1].set_title('ATAC', fontsize=20)

axes[0].set_yticklabels(axes[0].get_yticklabels(), fontsize = 20)
axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize = 20)

fig.suptitle('YS3 droplet type correlation', fontsize=24)

In [ ]:
all_donor_gex = pd.DataFrame(columns=tmp_gex_methods, index=tmp_gex_methods)
for method1 in tmp_gex_methods:
    mask1 = status_gex[method1] != 'non_singlet'
    tmp_gex = status_gex[mask1]
    for method2 in tmp_gex_methods:
        mask2 = tmp_gex[method2] != 'non_singlet'
        tmp_gex = tmp_gex[mask2]
        n_drops = tmp_gex.shape[0]
        all_donor_gex.loc[method1,method2] = float(np.sum(tmp_gex[method1] == tmp_gex[method2]) / n_drops)
all_donor_gex

In [ ]:
all_donor_atac = pd.DataFrame(columns=tmp_atac_methods, index=tmp_atac_methods)
for method1 in tmp_atac_methods:
    mask1 = status_atac[method1] != 'non_singlet'
    tmp_atac = status_atac[mask1]
    for method2 in tmp_atac_methods:
        mask2 = tmp_atac[method2] != 'non_singlet'
        tmp_atac = tmp_atac[mask2]
        n_drops = tmp_atac.shape[0]
        all_donor_atac.loc[method1,method2] = float(np.sum(tmp_atac[method1] == tmp_atac[method2]) / n_drops)
all_donor_atac

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 14))  # 2 rows, 4 columns
plt.subplots_adjust(wspace=0.4)

all_donor_gex = all_donor_gex.astype(float)
all_donor_atac = all_donor_atac.astype(float)

sns.heatmap(all_donor_gex, annot=True, xticklabels='', vmin=0.5, vmax=1, cmap='bwr',
            annot_kws={'size': 17}, cbar=False, ax = axes[0])

sns.heatmap(all_donor_atac, annot=True, xticklabels='', vmin=0.5, vmax=1, cmap='bwr',
            annot_kws={'size': 17}, cbar=False, ax = axes[1])
axes[0].set_title('RNA', fontsize=20)
axes[1].set_title('ATAC', fontsize=20)

axes[0].set_yticklabels(axes[0].get_yticklabels(), fontsize = 20)
axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize = 20)

fig.suptitle('YS3 individual identity correlation', fontsize=30)

In [ ]:
status_gex['demuxlet'].value_counts()

In [ ]:
n_drops = status_gex.shape[0]
gex_singlets = pd.DataFrame(columns=['RNA singlet rate'])
for method in status_gex.columns:
    gex_singlets.loc[method] = (n_drops - np.sum(status_gex[method] == 'non_singlet')) / n_drops
gex_singlets

In [ ]:
n_drops = status_atac.shape[0]
atac_singlets = pd.DataFrame(columns=['ATAC singlet rate'])
for method in status_atac.columns:
    atac_singlets.loc[method] = (n_drops - np.sum(status_atac[method] == 'non_singlet')) / n_drops
atac_singlets

In [ ]:
final_singlets = gex_singlets.copy()
final_singlets['ATAC singlet rate'] = atac_singlets['ATAC singlet rate']
final_singlets.fillna(0, inplace=True)

In [ ]:
final_singlets